# Advances of the Project

## Title / Objective:
Reproducible Analysis of CMS Open Data: Search for Dark Matter in Association with Top Quarks
(Based on the CMS publication: “Search for dark matter produced in association with a single top quark or a top quark pair in proton–proton collisions at ($\sqrt s = 13 \TeV$”).

# Physics Motivation

The Large Hadron Collider (LHC) operates at center-of-mass energies sufficient to probe physics beyond the Standard Model (BSM). A primary goal of the LHC program is to identify the nature of **Dark Matter (DM)**. In many simplified models, DM candidates ($\chi$) interact with Standard Model particles via a new mediator ($\phi$ or $a$). Since these mediators often couple to mass (Yukawa-like coupling), the **top quark**—being the heaviest elementary particle—offers the most sensitive probe for direct DM production.


At the parton level, the interaction produces top quarks accompanied by a mediator, which subsequently decays into a pair of invisible DM particles ($\chi\bar{\chi}$). This results in a final state characterized by heavy flavor quarks and significant **Missing Transverse Momentum ($p_T^{\text{miss}}$)** due to the invisible particles.

The dominant production mechanisms considered are:

* **Gluon Fusion ($t\bar{t}$+DM):**
    $$gg \to t \bar{t}\,\phi \to t \bar{t} + \chi \bar{\chi}$$

* **Single Top Associated Production ($tW$+DM):**
    $$gb \to t W \phi \to t W + \chi \bar{\chi}$$

* **t-channel Production ($tj$+DM):**
    $$qq' \to t j \phi \to t j + \chi \bar{\chi}$$

###  From Partons to Detector Signatures

The experimental signature is determined by the decay of the top quarks ($t \to W b$). The $W$ boson decays either hadronically ($W \to q\bar{q}'$, BR $\approx 67\%$) or leptonically ($W \to \ell \nu$, BR $\approx 33\%$). Consequently, the analysis is categorized into orthogonal **channels** based on the number of isolated charged leptons ($e, \mu$) in the final state:

* **All-Hadronic (AH) Channel (0 leptons):**
    * *Signature:* Multiple jets, b-tags, and large $p_T^{\text{miss}}$.
    * *Pros/Cons:* Highest signal yield, but suffers from overwhelming QCD multijet background and difficult trigger thresholds.

* **Single-Lepton (SL) Channel (1 lepton):**
    * *Signature:* Exactly one isolated lepton, jets, b-tags, and $p_T^{\text{miss}}$.
    * *Pros/Cons:* The "Golden Channel." It offers an optimal balance, suppressing QCD significantly while retaining higher statistics than the dilepton channel.

* **Dilepton (DL) Channel (2 leptons):**
    * *Signature:* Two opposite-sign leptons, jets, and $p_T^{\text{miss}}$.
    * *Pros/Cons:* Cleanest signal region with very low background, but statistically limited by the low branching fraction.

### The Single-Muon Channel

In this workflow, we focus exclusively on the **Single-Muon ($1\mu$) channel**.

This choice is driven by both physics and pedagogical clarity. Experimentally, muons provide a cleaner signature than electrons in the CMS detector, with lower bremsstrahlung rates and robust identification efficiencies. By requiring exactly one high-$p_T$ muon, we can effectively eliminate the QCD background, leaving **$t\bar{t}$+Jets** and **$W$+Jets** as the primary backgrounds to model.

This strategy allows us to demonstrate a complete End-to-End analysis—from object selection to statistical interpretation—in a controlled environment where the interplay between the DM signal (high $p_T^{\text{miss}}$) and Standard Model backgrounds can be clearly studied.

After defining the objective of the project (Reproducible Analysis of CMS Open Data), we discussed in which data format to work — NanoAOD or MiniAOD. We decided to use NanoAOD, because it is lighter and optimized for analysis tasks.

In the most precise version, we work within cernbox/swan, but it can work in any Jupyter environment; the important packages are: uproot and awkward

## Single Lepton

by B.V

We import all the libraries that we are going to use.

In [ ]:
# --- Standard scientific stack -------------------------------------------
import os
import json
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import hist
import requests

# --- Scikit-HEP ----------------------------------------------------------
import awkward as ak
import uproot
import vector

vector.register_awkward()

# --- Project modules -----------------------------------------------------
# dpoa_io       : plumbing only (downloads, retries, parallelism). No physics.
# dpoa_datasets : physics inputs (samples, cross sections, luminosity), each
#                 value carrying its provenance.
import dpoa_io
import dpoa_datasets as dd
from dpoa_io import N_FILES

AH_SUFFIX = "_0lep_raw"
DATA_NAME = "met"

print(f"awkward {ak.__version__} | uproot {uproot.__version__} | vector {vector.__version__}")
print(f"N_FILES = {N_FILES} per dataset")


We will use the **dpoa_workshop_utilities** module to help you access the datasets. The functions it contains are:


The `nanoaod_filenames` is a dictionary with the urls to the file indexes of the root files for every dataset that we will use in the analysis.

The `pretty_print(fields, fmt='40s', require=None, ignore=None)` function allows you to print subsets of keys based on strings that you require or ignore. It will also format that output based on how many characters you want in a column (you are limited to 80 characters per line).

The `build_lumi_mask(lumifile, tree, verbose=False)` function helps you mask (select) the data that's collected from collisions.

In [ ]:

#-------------------------------

import dpoa_workshop
from dpoa_workshop import (
    nanoaod_filenames,
    get_files_for_dataset,
    pretty_print,
    build_lumi_mask
)
#-------------------------------------------


In the drafts related to the papers, the datasets used from 2016 are listed along with their run periods and corresponding luminosities. However, one must be careful with these values, because not all periods are available and the data format differs from the one originally used, as previously noted.”

> **Reference figure `dataset_2016.png` is not included in this repository.**
> It is a screenshot of a table from the paper and carries no physics; see `paper.pdf`.


> **Reference figure `MC_data.png` is not included in this repository.**
> It is a screenshot of a table from the paper and carries no physics; see `paper.pdf`.


# Building the Ntuple File Index

CMS Open Data provides **file index text files** (`file_index.txt`) for each dataset.  
These contain the actual XRootD paths to the NanoAOD `.root` files, along with metadata such as the number of events per file. Example line:

root://eospublic.cern.ch//eos/opendata/cms/mc/.../nano_1.root nevts=58293


The objective is to collect file paths from multiple URLs, organize them by dataset, and store them in a JSON file, while handling possible download errors so the program keeps running.


### To streamline the workflow:
- We define a function `download_files(url)` that fetches each `file_index.txt` via HTTP and extracts only the ROOT file paths.  
- We loop over all entries in `nanoaod_filenames` (the dictionary we built earlier) and collect the full list of ROOT files per dataset.  
- The result is stored in a new dictionary `ntuples`, which maps *dataset -- list of ROOT file paths*.  
- Finally, we save this as a JSON file (`ntuples.json`) for later reuse

In [ ]:
# Build the ntuple index. dd.build_registry() layers two corrections on the
# upstream registry (RUNBOOK PHYS-01, PHYS-06):
#   * 'Zvv' is REPLACED. Upstream it resolved to ggH_HToSSTo4l -- a long-lived
#     particle SIGNAL sample -- while carrying the Z->nunu cross section.
#     Z(nunu) is 29.5% of the all-hadronic background (note Table 12).
#   * adds ZJetsToNuNu (record 74910), TTTo2L2Nu, and the W+jets HT-600..2500+
#     tail, which is what survives pTmiss > 250 GeV.
ntuples = dd.build_registry(nanoaod_filenames)

with open("ntuples.json", "w") as f:
    json.dump(ntuples, f, indent=1)

print()
print(f"{len(ntuples)} datasets indexed.")



Since `ntuples.json` concatenates all files into a single flat list per dataset, the distinction between source URLs (which often represent different periods like **Run2016G** vs **Run2016H**) is lost in the final file.

**Why counting matters:**
By logging the number of files in each part (`len(paths)`), we establish a **positional map**:
1.  It helps us identify which specific period a root file belongs to based on its index in the merged list.
2.  It ensures we can verify that the total file count matches the sum of the individual periods (sanity check).

And we will download important files like the luminosity file.

In [ ]:
# Golden JSON: the list of certified good luminosity sections for 2016.
# (Replaces `!wget`, which is not available on every platform and re-downloaded
# the file on every run.)
lumi_file = dpoa_io.download_lumi_json()


Complete dictionary, in case it's possible to add all the events but it hasn't been found or verified yet...

In [ ]:


XSEC2_PB = {

    
    ##### TOP QUARK #############
    "ttbar-semileptonic": 364.35,  
    "t-channel-top": 136.02,       # ST t-channel top
    "t-channel-antitop": 80.95,    # ST t-channel antitop
    "tW-top": 35.85,               # ST tW top 
    
    #### Rare Top 
    "ttW": 0.2043,  #  to TTWJetsToLNu
    "ttZ": 0.2529,  # to TTZToLLNuNu
    
    ##### WJETS (HT Binned) 
    
    "WJets-HT70to100":    1372.0,
    "WJets-HT100to200":   1345.0, # (Check this value, it seems high to be binned, maybe it's LO)
    "WJets-HT200to400":   359.7,  
    "WJets-HT400to600":   48.91,  
    "WJets-HT600to800":   12.05,
    "WJets-HT800to1200":  5.501,  
    "WJets-HT1200to2500": 1.329,  
    "WJets-HT2500toInf":  0.0322,

    #######DRELL-YAN (DY)
    "DYJets-HT100to200": 147.40,
    "DYJets-HT200to400": 40.99,
    "DYJets-HT400to600": 5.678,
    
    ##### DIBOSON 
    "WW": 118.7, 
    "ZZ": 16.6,   
    "WZ": 47.13  
}


until while these datasets are being used

In [ ]:
# Cross sections [pb]. Moved to dpoa_datasets.XSEC_PB so each value carries
# its source in a comment next to it. Two were wrong (RUNBOOK PHYS-01):
#
#   Zvv           the sample itself was wrong; now ZJetsToNuNu_Zpt-200toInf
#                 _BPSFilter at 0.7333 pb   [XSDB, 2026-09-03]
#   DYJets-Zpt200 was 1.27 pb, actually 0.393 pb -- a factor 3.2
#                 [XSDB, UL16 row]
#
# Everything else comes from note Table 6.
XSEC_PB = dd.XSEC_PB

print(f"{'sample':<22} {'xsec [pb]':>12}")
print("-" * 36)
for k, v in sorted(XSEC_PB.items()):
    print(f"{k:<22} {v:>12.4f}")


---

This function builds the fileset used by the analysis. It reads the JSON inventory of ntuples, identifies which samples are data or MC, applies an optional file limit for fast debugging, and attaches minimal metadata such as the cross section and number of files. The output is a clean, ready-to-use dictionary that tells the processor exactly which datasets to run over and how they should be treated.

In [ ]:

def construct_fileset(ntuples_json="ntuples.json", limit=None, verbose=True):
    """
    Parses the input JSON inventory and assigns metadata (xsec, is_data).
    
    Args:
        ntuples_json (str): Path to the JSON file containing the file lists.
        limit (int or None): Max number of files to load per process. 
                             Useful for quick debugging (e.g., limit=1).
                             If None, loads all files (production mode).
        verbose (bool): If True, prints a summary table of loaded samples.
    
    Returns:
        dict: A dictionary structured for the processor (Coffea/UpRoot).
    """
    
    with open(ntuples_json) as f:
        info = json.load(f)

    fileset = {}
    
    if verbose:
        print(f"\n{'Name':30} {'Type':>6} {'N Files':>10} {'XSEC [pb]':>12}")
        print("-" * 65)

    for process_name, file_list in info.items():
        
        # Slicing handles None gracefully, but explicit check is clearer for readers
        if limit is not None:
            files_to_use = file_list[:limit]
        else:
            files_to_use = file_list

        #  IDENTIFY TYPE (DATA vs MC) 
        
        if process_name in XSEC_PB:
            is_data = False
            xsec_value = XSEC_PB[process_name]
            proc_type = "MC"
        else:
            is_data = True
            xsec_value = None  # Real data has no theoretical xsec here
            proc_type = "DATA"

        #### BUILD DICTIONARY 
        # Minimal metadata structure to keep it lightweight

        
        fileset[process_name] = {
            "files": files_to_use,
            "metadata": {
                "is_data": is_data,
                "xsec": xsec_value,
                "n_files_loaded": len(files_to_use)
            }
        }

        #  LOGGING ---
        if verbose:
            xsec_str = f"{xsec_value:.2f}" if xsec_value else "-"
            print(f"{process_name:30} {proc_type:>6} {len(files_to_use):>10} {xsec_str:>12}")
    
    return fileset


This line initializes the full list of datasets for the analysis. It loads all ntuples defined in *ntuples.json* (since `limit=None`) and prints a summary of the samples. The resulting `fileset` becomes the central input that tells the processor which data and MC files to process.

In [ ]:

fileset = construct_fileset(
    ntuples_json="ntuples.json",
    limit=None,      
    verbose=True
)


We are about to treat the data as a "black box" in the processing loop. But we need to verify the inputs first. We use uproot, which is a Python library that allows us to read CERN ROOT files directly, without needing C++.

In [ ]:

dataset = "met" ##just for training

for i, fpath in enumerate(fileset[dataset]["files"][:10]):
    print(f"{i+1:2d}. {fpath}")



In [ ]:
sample = "ttbar-semileptonic"

root_path = fileset[sample]["files"][0]
print("Open:", root_path)

# open_events() retries with exponential backoff: the public eospublic
# redirector intermittently drops connections, and a single transient failure
# should not abort the notebook.
events = dpoa_io.open_events(root_path)

print("# events:", events.num_entries)

all_keys = events.keys()
print(f"Total branches: {len(all_keys)}")


In [ ]:

if "genWeight" in events.keys():
    gw = events["genWeight"].array(entry_stop=100_000, library="np")
    print(f"genWeight: mean={gw.mean():.3f}, std={gw.std():.3f}, negativos={(gw<0).mean():.2%}")
else:
    print("There is no genWeight")



    genWeight: mean=300.828, std=39.097, negativos=0.42%


Because std > 0 and negativos > 0, you have learned a golden rule for this dataset:

NEVER count events.

Wrong: N = len(events)

Right: N = sum(events.genWeight)

If you just count them (len), you will be wrong by 0.84% (counting the negatives as positives instead of subtracting them).

---
## "Having defined the dataset, we move to data processing. Since the reference analysis has already optimized the kinematic variables, we will adopt their final cuts directly. We will retrieve these specific thresholds from the summary tables and apply the necessary selection techniques to our data."
---



**The workflow proceeds in three main steps:**

1. **Apply the baseline selection**
   We first select events that match the single-lepton topology (electron or muon).
   This removes events that clearly do not belong to the channel and defines the core event structure we will work with.

2. **Filter the events**
   After the baseline, we keep only the events that satisfy these basic criteria.
   This produces a clean and well-defined sample for further analysis.

3. **Apply additional cuts to build CR and SR**
   With the filtered sample, we introduce extra requirements to define:

   * **Control Regions (CR):** used to study and validate the dominant backgrounds.
   * **Signal Region (SR):** where the dark-matter search is performed.

This step-by-step structure mirrors the logic used in the CMS paper and ensures a transparent and reproducible analysis flow.


**So all of this is reflected in the tables, which are directly the cuts we should use, 
and we don't have to explore it ourselves in this case for this guide.**

### Base line


If you read the drafts


#### **Table 10 — Baseline Selection (SL and AH)**


The **minimum or baseline selection** that every event must satisfy **before being assigned to any region** (Signal or Control).

It includes:

* Lepton quality and type (muons, electrons)
* Minimum cuts on **pT** and **η**
* Requirements on **jets** and **b-jets**
* Event-cleaning criteria
* **Additional-lepton veto**

It acts as the **universal first filter** of the analysis.
No event enters the SR or CR without passing this baseline.

---

#### **Tables 11 and 12 — Expected Events After Optimized Cuts (SL and AH)**

Tables showing:

* Expected event yields for each **background**
* Predictions for different **signal models**
* Results after the baseline plus **optimized cuts**


These tables serve **only for internal validation** in the paper:

* Checking consistency between simulated backgrounds and signals
* Showing how many events remain under different signal hypotheses

> **Reference figure `Table_10.png` is not included in this repository.**
> It is a screenshot of a table from the paper and carries no physics; see `paper.pdf`.


In [ ]:
vector.register_awkward()

# --- b tagging (RUNBOOK PHYS-08) ------------------------------------------
# DeepJet medium working point. Was 0.2770, which is the 2017 value; this data
# is UL2016 and Run2016H is postVFP, whose medium WP is 0.2489.
# (The paper itself uses CSVv2 medium; NanoAOD exposes it as Jet_btagCSVV2.
#  Switching discriminator changes efficiency and mistag rate, so that is left
#  as an open physics decision -- see RUNBOOK PHYS-08.)
BTAG_WP_DEEPJET_MEDIUM_UL16POSTVFP = 0.2489   # UL16 postVFP
BTAG_WP_CSVV2_MEDIUM_2016          = 0.8484   # what the paper actually uses

# The paper tags b jets with CSVv2 medium (note Table 10, "CSVM"). We switch to
# it because the choice matters much more than it looks:
#
#   Z(nunu)+jets and W+jets contain almost no real b quarks, so they enter the
#   n_b >= 1 region almost entirely through MISTAGGED light jets. DeepJet has a
#   markedly lower light-jet mistag rate than CSVv2, so using DeepJet suppresses
#   exactly those two processes while leaving ttbar (real b jets, similar
#   efficiency in both taggers) almost untouched.
#
#   That is precisely the distortion we observe: Z(nunu) at 4.5% instead of the
#   29.7% of note Table 12, with ttbar correspondingly inflated.
#
# Both counts are stored so the choice can be revisited without reprocessing.
BTAG_DISCRIMINANT = "csv"


# --- top pT reweighting (RUNBOOK PHYS-14), note section "Corrections for MC"
# The generated top pT spectrum in ttbar MC is harder than what is observed.
# The note gives the correction explicitly:
#
#     rho(pT) = exp(0.0615 - 0.0005 * pT)          pT at matrix-element level
#     w       = sqrt(rho(pT_1) * rho(pT_2))
#
# It is normalisation-preserving by construction (measured <w> = 1.0005 over
# all events) but NOT shape-preserving: measured <w> = 0.892 for pTmiss > 250,
# because the signal regions select the boosted tail the correction suppresses.
#
# The two generated tops are taken as the last copies (statusFlags bit 13).
# Using the hard-process copies (bit 7) instead gives the same weight to four
# decimal places, and both give exactly two tops in 100 % of events.
TOP_PT_LASTCOPY_BIT = 13


def top_pt_weight(tree, n_events):
    """Per-event sqrt(rho1*rho2); 1.0 for any event without exactly two tops."""
    w = np.ones(n_events)
    if "GenPart_pdgId" not in tree.keys():
        return w
    pdg = tree["GenPart_pdgId"].array()
    sf  = tree["GenPart_statusFlags"].array()
    pt  = tree["GenPart_pt"].array()
    mask = (abs(pdg) == 6) & ((sf & (1 << TOP_PT_LASTCOPY_BIT)) != 0)
    top_pt = pt[mask]
    rho = np.exp(0.0615 - 0.0005 * top_pt)
    ww = ak.to_numpy(ak.fill_none(np.sqrt(ak.prod(rho, axis=1)), 1.0))
    ok = ak.to_numpy(ak.num(top_pt) == 2)
    w[ok] = ww[ok]
    return w

# --- MET noise filters (RUNBOOK PHYS-07) ----------------------------------
# note section 2.1. These remove INSTRUMENTAL pTmiss -- fake missing energy
# from detector noise. None were applied upstream, and every event they should
# have removed sat in the data histogram at high pTmiss, which is exactly the
# region these plots show.
MET_FILTERS = [
    "Flag_goodVertices",
    "Flag_HBHENoiseFilter",
    "Flag_HBHENoiseIsoFilter",
    "Flag_EcalDeadCellTriggerPrimitiveFilter",
    "Flag_globalTightHalo2016Filter",
    "Flag_BadPFMuonFilter",
    "Flag_BadChargedCandidateFilter",   # NanoAOD name for the note's
                                        # BadChargedHadronFilter
]
MET_FILTERS_DATA_ONLY = ["Flag_eeBadScFilter"]


def process_file_0lep(filename, dataset="Unknown", IS_DATA=False):
    """All-hadronic (0-lepton) baseline selection - note Table 10.

    Baseline required by note Table 10:
        nlep = 0 (e or mu, pT > 10 GeV)
        njet >= 3   (pT > 30 GeV)
        nb   >= 1
        pTmiss >= 250 GeV
        min dphi(j1,2, pTmiss) > 0.4

    Changes from the upstream function, all in RUNBOOK Part 2:
      PHYS-03  nb >= 1 was never applied; pTmiss was 240, not 250;
               min dphi > 0.4 was never applied
      PHYS-04  min dphi was minimised over the top FOUR jets; the paper uses
               the two leading jets, so the old cut was systematically tighter
      PHYS-05  leading-jet pT is now stored so pT(j1)/HT can be cut on
      PHYS-07  MET noise filters and the golden-JSON lumi mask now applied
      PHYS-08  b-tag working point 0.2770 -> 0.2489
    """
    with dpoa_io.open_events(filename) as tree:
        muons = ak.zip({
            "pt":      tree["Muon_pt"].array(),
            "eta":     tree["Muon_eta"].array(),
            "phi":     tree["Muon_phi"].array(),
            "mass":    tree["Muon_mass"].array(),
            "iso":     tree["Muon_pfRelIso04_all"].array(),
            "looseId": tree["Muon_looseId"].array(),
        }, with_name="Momentum4D")

        electrons = ak.zip({
            "pt":       tree["Electron_pt"].array(),
            "eta":      tree["Electron_eta"].array(),
            "phi":      tree["Electron_phi"].array(),
            "mass":     tree["Electron_mass"].array(),
            "cutBased": tree["Electron_cutBased"].array(),
        }, with_name="Momentum4D")

        jets = ak.zip({
            "pt":    tree["Jet_pt"].array(),
            "eta":   tree["Jet_eta"].array(),
            "phi":   tree["Jet_phi"].array(),
            "mass":  tree["Jet_mass"].array(),
            "jetId": tree["Jet_jetId"].array(),
            "btag":  tree["Jet_btagDeepFlavB"].array(),
            "csv":   tree["Jet_btagCSVV2"].array(),
        }, with_name="Momentum4D")

        met_pt  = tree["MET_pt"].array()
        met_phi = tree["MET_phi"].array()
        gen_weight = tree["genWeight"].array() if not IS_DATA else ak.ones_like(met_pt)
        # PHYS-14: only ttbar has generated tops; every other sample gets 1.
        top_w = ak.Array(top_pt_weight(tree, len(met_pt))
                         if (not IS_DATA and dataset.startswith("ttbar"))
                         else np.ones(len(met_pt)))

        # --- PHYS-07: MET noise filters --------------------------------
        filters = list(MET_FILTERS) + (MET_FILTERS_DATA_ONLY if IS_DATA else [])
        pass_filters = ak.ones_like(met_pt, dtype=bool)
        for flag in filters:
            pass_filters = pass_filters & tree[flag].array()

        # --- PHYS-07: golden JSON, data only ---------------------------
        if IS_DATA:
            pass_lumi = build_lumi_mask(str(dpoa_io.LUMI_JSON), tree)
        else:
            pass_lumi = ak.ones_like(met_pt, dtype=bool)

    # ----- LEPTON VETO (note Table 10: nlep = 0, pT > 10) -----------------
    mu_veto_mask  = (muons.pt > 10) & (abs(muons.eta) < 2.4) & (muons.looseId) & (muons.iso < 0.25)
    ele_veto_mask = (electrons.pt > 10) & (abs(electrons.eta) < 2.5) & (electrons.cutBased >= 1)

    pass_lepton_veto = (ak.num(muons[mu_veto_mask]) == 0) & (ak.num(electrons[ele_veto_mask]) == 0)

    # PHYS-03: 250 GeV, per note Table 10 (was 240)
    pass_met = met_pt >= 250

    event_mask = pass_lepton_veto & pass_met & pass_filters & pass_lumi

    jets       = jets[event_mask]
    met_pt     = met_pt[event_mask]
    met_phi    = met_phi[event_mask]
    gen_weight = gen_weight[event_mask]
    top_w      = top_w[event_mask]                 # PHYS-14: same mask

    # ----- JET CLEANING ---------------------------------------------------
    jet_clean_mask = (jets.pt > 30) & (jets.jetId >= 1)
    all_clean_jets = jets[jet_clean_mask]

    central_jets = all_clean_jets[abs(all_clean_jets.eta) < 2.4]
    forward_jets = all_clean_jets[(abs(all_clean_jets.eta) >= 2.4) & (abs(all_clean_jets.eta) < 5.0)]

    has_3_central = ak.num(central_jets) >= 3

    f_met        = met_pt[has_3_central]
    f_met_phi    = met_phi[has_3_central]
    f_central    = central_jets[has_3_central]
    f_forward    = forward_jets[has_3_central]
    f_gen_weight = gen_weight[has_3_central]
    f_top_w      = ak.to_numpy(top_w[has_3_central])   # PHYS-14

    # ----- KINEMATIC VARIABLES -------------------------------------------
    # PHYS-04: the TWO leading jets (note Table 13), not the top four.
    jets_top2 = f_central[:, :2]
    dphi_jets = np.mod(abs(jets_top2.phi - f_met_phi) + np.pi, 2 * np.pi) - np.pi
    min_dphi  = ak.min(abs(dphi_jets), axis=1)

    HT      = ak.sum(f_central.pt, axis=1)
    met_sig = f_met / np.sqrt(HT)

    # PHYS-05: leading jet pT, for the pT(j1)/HT cut in the nb >= 2 region
    jet1_pt = f_central[:, 0].pt

    # PHYS-08: the paper's tagger (CSVv2 medium). Both are stored.
    is_btag_deepjet = f_central.btag > BTAG_WP_DEEPJET_MEDIUM_UL16POSTVFP
    is_btag_csv     = f_central.csv  > BTAG_WP_CSVV2_MEDIUM_2016
    is_btag = is_btag_csv if BTAG_DISCRIMINANT == "csv" else is_btag_deepjet
    n_btag  = ak.sum(is_btag, axis=1)

    # mT_b: transverse mass of pTmiss and the leading b-tagged jet
    b_jets       = f_central[is_btag]
    has_b        = ak.num(b_jets) > 0
    b_jet_padded = ak.fill_none(ak.firsts(b_jets), {"pt": 0, "phi": 0}, axis=0)
    dphi_b       = b_jet_padded.phi - f_met_phi
    calc_mt_b    = np.sqrt(2 * b_jet_padded.pt * f_met * (1 - np.cos(dphi_b)))
    mt_b_np      = ak.to_numpy(calc_mt_b)
    mt_b_np[~ak.to_numpy(has_b)] = -1.0

    n_fwd = ak.num(f_forward)

    out = {
        "dataset":       dataset,
        "met":           ak.to_numpy(f_met),
        "HT":            ak.to_numpy(HT),
        "jet1_pt":       ak.to_numpy(jet1_pt),
        "met_sig":       ak.to_numpy(met_sig),
        "nJet":          ak.to_numpy(ak.num(f_central)),
        "nBTag":         ak.to_numpy(n_btag),
        "nBTag_deepjet": ak.to_numpy(ak.sum(is_btag_deepjet, axis=1)),
        "nBTag_csv":     ak.to_numpy(ak.sum(is_btag_csv, axis=1)),
        "min_dphi":      ak.to_numpy(min_dphi),
        "nForwardJets":  ak.to_numpy(n_fwd),
        "mT_b":          mt_b_np,
        "genWeight":     ak.to_numpy(f_gen_weight),
        "topPtWeight":   f_top_w,
    }

    # ----- BASELINE, completed (note Table 10) ----------------------------
    # PHYS-03: nb >= 1 and min dphi > 0.4 were both missing upstream. They are
    # exactly the cuts that remove QCD multijet and instrumental pTmiss, so
    # omitting them inflated DATA far more than MC.
    df = pd.DataFrame(out)
    baseline = (df["nBTag"] >= 1) & (df["min_dphi"] > 0.4)
    return df[baseline].reset_index(drop=True)


In [ ]:
#f = 'root://eospublic.cern.ch//eos/opendata/cms/mc/RunIISummer20UL16NanoAODv9/TTToSemiLeptonic_TuneCP5_13TeV-powheg-pythia8/NANOAODSIM/106X_mcRun2_asymptotic_v17-v1/120000/08FCB2ED-176B-064B-85AB-37B898773B98.root'

In [ ]:
#process_file_muon(f,'ttbar-semileptonic',IS_DATA=False)

This section details the three-stage pipeline designed to ingest raw ROOT files and convert them into manageable Parquet datasets for the **Single-Muon Channel**.

#### 🔹 BLOCK 1: Dataset Configuration
Defines the scope of the analysis for the muon channel.

* **Classification:** It strictly separates the input streams into:
    * **Data:** `SingleMuon` (Real experimental data).
    * **Monte Carlo (MC):** All other datasets ($t\bar{t}$, $W$+Jets, etc.).
* **Utility:** This separation allows the downstream logic to automatically determine if **Generator Weights** (required for MC normalization) should be extracted or ignored.

#### 🔹 BLOCK 2: The Orchestrator Function
 Acts as the high-level **Manager Layer**. It abstracts the file I/O and parallelization logic from the physics analysis.
 It does *not* perform physics calculations. Its role is strictly operational: **Loop $\rightarrow$ Collect $\rightarrow$ Assemble $\rightarrow$ Save**.
* **Workflow Steps:**
    1.  **Validation:** Verifies the dataset exists in the global `fileset` registry.
    2.  **Differentiation:** Sets the `is_data` flag based on the dataset name.
    3.  **Iteration:** Loops through the first `n_files` of the requested dataset.
    4.  **Delegation:** Calls the physics kernel (`process_file_muon`) to extract observables from each file.
    5.  **Aggregation:** Concatenates partial results into a unified `pandas.DataFrame`.
    6.  **Persistence:** Serializes the final DataFrame into a binary **Parquet** file in the `output_raw/` directory.

#### 🔹 BLOCK 3: Execution Loop
 Triggers the batch processing.
* **Function:** Iterates through the list defined in Block 1.
* **Control:** It calls the orchestrator for each dataset, enforcing a limit on the number of input files (`N_FILES`) to control the sample size during testing or production runs.

In [ ]:
# MC samples for the all-hadronic channel.
# Added vs upstream (RUNBOOK PHYS-01, PHYS-06), with their share of the
# background from note Table 12:
#     Zvv-Zpt200          Z(nunu), 29.5%  -- upstream pointed at the wrong file
#     ttbar-2l2nu         tt(2l),   3.8%
#     WJets-HT600..2500+  part of W+jets, 15.1%
datasets_AH_mc = [
    'ttbar-semileptonic',   # tt(1l), 41.7% of the background
    'ttbar-2l2nu',          # tt(2l),  3.8%
    'Zvv-Zpt200',           # Z(nunu) high-pT bin
    'Zvv-Zpt100to200',      # Z(nunu) low-pT bin: MET > 250 still receives
                            # events from here through jet mismeasurement
                            # and extra jet recoil, so the bin is not empty.
    'WJets-HT70to100', 'WJets-HT100to200', 'WJets-HT200to400',
    'WJets-HT400to600', 'WJets-HT600to800', 'WJets-HT800to1200',
    'WJets-HT1200to2500', 'WJets-HT2500toInf',
    't-channel-top', 't-channel-antitop',
    'ttW', 'WW', 'ZZ', 'WZ', 'DYJets-Zpt200',
]

print(f"=== ALL-HADRONIC - MC ({N_FILES} files per dataset) ===")

ah_reports = {}
for ds in datasets_AH_mc:
    _, rep = dpoa_io.process_dataset(
        ds, fileset, process_file_0lep, is_data=False,
        n_files=N_FILES, suffix=AH_SUFFIX,
    )
    ah_reports[ds] = rep


**Note:**

Notice that we could reuse the dictionary built earlier (`fileset`) to automatically loop over all available datasets. However, for debugging and flexibility, a manual list of datasets was defined. At the beginning, only a few datasets were used for quick tests, and later this list was expanded to match the same datasets contained in the `fileset`. This approach allows easier control during development while keeping compatibility with the full processing workflow.

In [ ]:
# Real data: the MET primary dataset.
print(f"=== ALL-HADRONIC - DATA ({N_FILES} files) ===")

_, rep = dpoa_io.process_dataset(
    "met", fileset, process_file_0lep, is_data=True,
    n_files=N_FILES, suffix=AH_SUFFIX,
)
ah_reports["met"] = rep



Runs/genEventSumw


This block reads that information *efficiently* from the `Runs` tree in each file.

### **What this block does for normalization**

* Provides **N_gen ≡ sumGenWeights**
* Without it, normalization would be impossible
* For **data**, you correctly set the weight to **1** (data is never scaled)

In [ ]:
# Sum of generator weights, needed to normalise MC:  w = xsec * L / sumGenWeights
#
# This MUST cover exactly the same files that were processed above. In the
# original notebook the processing used N_FILES and this cell used a separate
# N_FILES_MC literal; when the two drifted apart every MC sample was normalised
# by the wrong denominator. Both now read the single N_FILES constant.
DATA_SL = {"met"}

sum_weights_map = dpoa_io.sum_gen_weights(fileset, DATA_SL, n_files=N_FILES)


# Defining the Physics Metadata (Cross Sections + Classification)


### **Explanation**

This block assigns:

* **Cross sections (σ)**
* **Dataset category (ttbar, WJets, etc.)**
* **Plotting style (colors)**

This is where you get the numerator of the weight formula:

$$
\sigma \cdot L
$$

### **How it connects to normalization**

You now have:

* **σ** (from `fileset[ds]["metadata"]`)
* **L** (your chosen `LUM`)
* **N_gen** (from previous block)

→ Everything needed to compute **per-event scale factors**.

In [ ]:
# --- Luminosity (RUNBOOK PHYS-02) ----------------------------------------
# Upstream used a single LUM = 3400 pb^-1 for every channel, chosen to make the
# plots agree rather than derived. Luminosity is a property of WHICH data
# dataset was processed and HOW MUCH of it, so one constant cannot be right for
# two channels.
#
# Open Data exposes only Run2016H = 8.9 fb^-1 (note Table 2). dd.luminosity_pb
# measures the processed fraction by bytes of the record -- exact metadata, and
# a better proxy than file count since file sizes vary by more than 2x.
DATA_SL = {DATA_NAME}

LUM = dd.luminosity_pb(DATA_NAME, fileset[DATA_NAME]["files"][:N_FILES])
print(f"\nUsing L = {LUM:.1f} pb^-1   (upstream used 3400.0)")

# Colours sampled pixel-by-pixel from Figure 5 of paper.pdf so the stacks are
# directly comparable. The upstream palette used different colours for every
# process, which made a side-by-side against the paper meaningless.
COLOR_MAP = {
    "ttbar":   "#FFCC33",   # paper: yellow
    "Zvv":     "#6699FF",   # paper: light blue
    "WJets":   "#9933FF",   # paper: purple
    "tX":      "#FF9933",   # paper: orange
    "Zll":     "#009900",   # paper: green
    "Diboson": "#000099",   # paper: dark blue
}


def get_group_info(name):
    """Grouping as in the paper legend: single top and ttV are folded into
    t+X, and Z(nunu) is kept separate from Z(ll)."""
    if "Zvv" in name:                     return "Zvv", r"Z($\nu\nu$) + jets", COLOR_MAP["Zvv"]
    if "DY" in name:                      return "Zll", r"Z($\ell\ell$) + jets", COLOR_MAP["Zll"]
    if "ttbar" in name:                   return "ttbar", r"$t\bar{t}$", COLOR_MAP["ttbar"]
    if "WJets" in name:                   return "WJets", r"W($\ell\nu$) + jets", COLOR_MAP["WJets"]
    if "channel" in name or "tW" in name: return "tX", r"t+X", COLOR_MAP["tX"]
    if "ttW" in name or "ttZ" in name:    return "tX", r"t+X", COLOR_MAP["tX"]
    if name in ["WW", "ZZ", "WZ"]:        return "Diboson", r"VV, VH", COLOR_MAP["Diboson"]
    return "tX", r"t+X", COLOR_MAP["tX"]


def scale_factor(dataset):
    """MC event weight scale:  xsec * L / sumGenWeights. None for data."""
    if dataset in DATA_SL:
        return None
    xsec = fileset[dataset]["metadata"]["xsec"]
    sum_w = sum_weights_map.get(dataset, 0.0)
    if not sum_w or xsec is None:
        return None
    return (xsec * LUM) / sum_w


print(f"\n{'Dataset':<22} | {'Xsec [pb]':>10} | {'SumGenW':>12} | {'Scale':>11}")
print("-" * 66)
for ds in fileset:
    if ds in DATA_SL:
        print(f"{ds:<22} | {'-':>10} | {'-':>12} | {'1.00':>11}")
        continue
    xsec = fileset[ds]["metadata"]["xsec"]
    sum_w = sum_weights_map.get(ds, 0.0)
    sc = scale_factor(ds)
    if sc is not None:
        print(f"{ds:<22} | {xsec:>10.4f} | {sum_w:>12.3e} | {sc:>11.3e}")
    else:
        print(f"{ds:<22} | {str(xsec):>10} | {sum_w:>12.3e} | {'--':>11}")


#### Regarding the luminosity definition, we employ a fractional approach calculated as `LUM = (20/82) * 8900.0`(perdiod H) .This method posits that processing 20 out of 82 files accounts for roughly  of the data. While strictly an estimate—given that individual ROOT files in CMS datasets vary in event content and processing a fixed number of files does not yield a precise linear fraction of the total luminosity—it aligns sufficiently with the observed yields, representing approximately  of the Run 2016 dataset. We intentionally maintain this minimalist strategy to serve as a guide for reproducibility; it allows the analysis to be run efficiently on personal computers for educational purposes, preserving the correct physics structure without the significant overhead of processing the full multi-terabyte dataset.



To produce publication-quality plots—like those found in CMS analyses—we need a function that:

1. **Builds histograms for each MC process**, applying the final per-event normalization weight.
2. **Groups MC samples by physics category** (tt̄, W+jets, Single Top, Diboson, etc.).
3. **Stacks the MC contributions** in the correct order (smallest on top) so that the final shape reflects the Standard Model expectation.
4. **Overlays the real DATA** histogram with error bars.
5. **Applies CMS styling**, including the “Preliminary” tag and integrated luminosity.
6. Ensures the plot is clean, readable, and suitable for analysis notes or a professional report.

This function does exactly that.
By defining it once, we ensure that **every observable (pT, eta, MET, jets, etc.)** is plotted consistently and in the same professional format used in CMS.

---

In [ ]:
import mplhep as hep
import matplotlib.pyplot as plt

plt.style.use(hep.style.CMS)


def plot_grouped_stack(var_name, x_label, x_range, channel_data="met",
                       n_bins=30, log_scale=False, dfs=None):
    """Stacked MC vs data histogram.

    `dfs` was previously read from a module-level `all_dfs` global. It is now
    an explicit argument, so re-running an earlier cell cannot silently plot a
    different set of dataframes; it still falls back to `all_dfs` so existing
    calls keep working.
    """
    if dfs is None:
        dfs = all_dfs

    bins = np.linspace(x_range[0], x_range[1], n_bins + 1)

    grouped_counts = {}
    grouped_info = {}

    for name, df in dfs.items():
        if name in DATA_SL:
            continue

        group_key, label, color = get_group_info(name)
        counts, _ = np.histogram(df[var_name], bins=bins, weights=df["final_weight"])

        if group_key not in grouped_counts:
            grouped_counts[group_key] = counts
            grouped_info[group_key] = {"label": label, "color": color, "yield": np.sum(counts)}
        else:
            grouped_counts[group_key] += counts
            grouped_info[group_key]["yield"] += np.sum(counts)

    # Order groups by yield: smallest on top of the stack
    active_groups = sorted(grouped_info, key=lambda g: grouped_info[g]["yield"])

    mc_counts, mc_colors, mc_labels = [], [], []
    total_mc = np.zeros(n_bins)

    for g in active_groups:
        mc_counts.append(grouped_counts[g])
        mc_colors.append(grouped_info[g]["color"])
        mc_labels.append(grouped_info[g]["label"])
        total_mc += grouped_counts[g]

    df_data = dfs.get(channel_data)
    if df_data is None:
        print(f"ERROR: No DATA found for channel {channel_data}")
        return

    data_counts, _ = np.histogram(df_data[var_name], bins=bins)

    fig, ax = plt.subplots(figsize=(10, 8))

    if mc_counts:
        hep.histplot(mc_counts, bins=bins, stack=True, histtype="fill",
                     color=mc_colors, label=mc_labels,
                     edgecolor="black", linewidth=1, ax=ax)

    hep.histplot(data_counts, bins=bins, histtype="errorbar", color="black",
                 label=f"{channel_data} (Data)", yerr=True,
                 marker="o", markersize=5, ax=ax)

    hep.cms.label("Preliminary", data=True, lumi=round(LUM / 1000.0, 1), year=2016, ax=ax)

    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles[::-1], labels[::-1], fontsize=16, ncol=2, loc="upper right")
    ax.set_xlabel(x_label, fontsize=24)
    ax.set_ylabel("Events", fontsize=24)
    ax.set_xlim(x_range)

    ceiling = max(np.max(data_counts), np.max(total_mc), 1.0)
    if log_scale:
        ax.set_yscale("log")
        ax.set_ylim(0.1, ceiling * 500)
    else:
        ax.set_ylim(0, ceiling * 1.5)

    plt.tight_layout()
    plt.show()


This block loads every muon-channel dataset from the previously generated _raw.parquet files, applies the correct normalization (data vs. Monte Carlo), assigns a final per-event weight, and stores everything inside a unified dictionary (all_dfs).
Once all datasets are normalized, we can safely call plot_grouped_stack to generate stacked MC vs. Data plots that follow a professional CMS-style presentation.

In [ ]:
def load_channel():
    """Load the all-hadronic parquet files and attach event weights."""
    datasets = [DATA_NAME] + datasets_AH_mc
    loaded, missing = {}, []

    for ds in datasets:
        path = dpoa_io.OUTPUT_DIR / f"{ds}{AH_SUFFIX}.parquet"
        if not path.exists():
            missing.append(ds)
            continue
        df = pd.read_parquet(path)
        if ds in DATA_SL:
            df["final_weight"] = 1.0
        else:
            sc = scale_factor(ds)
            if sc is None:
                missing.append(f"{ds} (no scale factor)")
                continue
            w = df["genWeight"] * sc if "genWeight" in df else sc
            # PHYS-14: top pT reweighting. Older parquets predate the column,
            # so its absence means "no correction", not an error.
            if "topPtWeight" in df:
                w = w * df["topPtWeight"]
            df["final_weight"] = w
        loaded[ds] = df

    print(f"Loaded {len(loaded)} datasets.")
    if missing:
        print(f"NOT loaded ({len(missing)}): {', '.join(missing)}")
    return loaded


olep_dfs_clean = load_channel()
all_dfs = olep_dfs_clean

# --- Composition check against note Table 12 -----------------------------
# This is the benchmark for the whole exercise: after the AH baseline the note
# gives tt(1l) 41.7%, Z(nunu) 29.5%, W+jets 15.1%, single top 5.1%,
# tt(2l) 3.8%, VV 2.7%, QCD 1.1%, ttV 0.9%, Z(ll) 0.2%, and data/MC = 1.06.
TABLE12 = {"ttbar": 45.5, "ZJets": 29.7, "WJets": 15.1,
           "SingleTop": 5.1, "Diboson": 2.7, "Rare": 0.9}

if olep_dfs_clean:
    yields = {}
    for name, df in olep_dfs_clean.items():
        if name in DATA_SL:
            continue
        g, _, _ = get_group_info(name)
        yields[g] = yields.get(g, 0.0) + float(df["final_weight"].sum())

    total_mc = sum(yields.values())
    n_data = len(olep_dfs_clean[DATA_NAME])

    print(f"\n{'group':<12} {'events':>12} {'obs %':>8} {'Table 12 %':>12}")
    print("-" * 48)
    for g in sorted(yields, key=lambda k: -yields[k]):
        pct = 100 * yields[g] / total_mc if total_mc else 0
        ref = TABLE12.get(g)
        print(f"{g:<12} {yields[g]:>12.1f} {pct:>7.1f}% "
              f"{(f'{ref:.1f}%' if ref else '-'):>12}")
    print("-" * 48)
    print(f"{'total MC':<12} {total_mc:>12.1f}")
    print(f"{'data':<12} {n_data:>12d}")
    print(f"\ndata/MC = {n_data / total_mc:.3f}   (note Table 12: 1.06)")

    plot_grouped_stack("met",   r"$p_T^{miss}$ [GeV]", (250, 500), DATA_NAME, dfs=all_dfs, log_scale=True)
    plot_grouped_stack("nJet",  r"$N_{jets}$",         (2, 10),    DATA_NAME, dfs=all_dfs, n_bins=8, log_scale=True)
    plot_grouped_stack("nBTag", r"$N_{b-tags}$",       (0, 5),     DATA_NAME, dfs=all_dfs, n_bins=5, log_scale=True)
else:
    print("No data loaded. Run the processing cells above first.")


and with the same idea as the muon one

> **Reference figure `Table_13.png` is not included in this repository.**
> It is a screenshot of a table from the paper and carries no physics; see `paper.pdf`.


In [ ]:
# `load_data` is kept as a thin alias so the signal- and control-region cells
# below read exactly as they do in the documentation. The suffix now points at
# the all-hadronic parquet files rather than the single-lepton ones.
def load_data(channel="met"):
    return load_channel()


In [ ]:
def signal_regions(df):
    """All-hadronic signal regions - note Table 13.

    Table 13 (AH columns), on top of the baseline already applied:
        pTmiss                >= 250 GeV
        min dphi(j1,2)        >= 1.0
        mT_b                  >= 180 GeV
        pT(j1)/HT             <= 0.5     (nb >= 2 region ONLY)
        categories: nb=1 & 0 FJ | nb=1 & >=1 FJ | nb>=2

    RUNBOOK PHYS-05: mT_b was computed upstream but never cut on, and
    pT(j1)/HT was never computed at all.
    """
    if df is None or len(df) == 0:
        return {}

    pass_common = (
        (df["met"] >= 250) &
        (df["min_dphi"] >= 1.0) &
        (df["mT_b"] >= 180)          # PHYS-05: was computed, never applied
    )

    df_sr = df[pass_common]
    if len(df_sr) == 0:
        return {}

    mask_1b_0f = (df_sr["nBTag"] == 1) & (df_sr["nForwardJets"] == 0)
    mask_1b_1f = (df_sr["nBTag"] == 1) & (df_sr["nForwardJets"] >= 1)
    # PHYS-05: pT(j1)/HT <= 0.5 applies to the nb >= 2 region only
    mask_2b    = (df_sr["nBTag"] >= 2) & (df_sr["jet1_pt"] / df_sr["HT"] <= 0.5)

    return {
        "SR_1b_0f": df_sr[mask_1b_0f],
        "SR_1b_1f": df_sr[mask_1b_1f],
        "SR_2b":    df_sr[mask_2b],
    }


Once `plot_grouped_stack` was defined, the same base code was reused to create `plot_region_stack`. The logic of the stacked histogram is the same, but extra features were added to handle analysis regions. In particular, a region label is drawn on the plot and the display is adapted to show results specific to a selected region. This keeps the core behavior unchanged while adding more flexibility for regional visualization.


The main changes are:

* A region label parameter was added to the function signature.
* A text label was drawn on the plot using a line like:
  `ax.text(..., region_label, ...)`
* The plot title or decoration was adapted to reflect the selected region.
* Small adjustments were made to make the function work with region-specific dictionaries.

All other parts of the stacking logic (histogram creation, grouping, colors, and legends) remain unchanged.

In [ ]:
from collections import OrderedDict


# --- stacking order, RUNBOOK PLOT-01 --------------------------------------
# The paper stacks processes in a FIXED physics order (bottom to top), not by
# yield. Sorting by yield puts the largest process at the bottom, which for the
# single-lepton channel (ttbar dominant) inverts the picture completely: the
# same numbers produce a stack that looks nothing like Figure 4.
# Read off the legends of Figures 4 and 5, bottom of the stack first:
PAPER_STACK_ORDER = ["Diboson", "tX", "ttbar", "WJets", "Zvv", "Zll"]


def plot_region_stack(dfs_dict, region_name, var_name, x_label, x_range,
                      channel_label, n_bins=10, y_span=(1000.0, 150.0),
                      overflow=True, lumi_label=None):
    """Stacked MC vs data with a Data/Bkg ratio panel underneath.

    Laid out like Figures 4 and 5 of the paper: log-scale main panel, ratio
    panel below with a line at 1. The upstream version had no ratio panel,
    which is the panel that actually shows whether data and simulation agree.

    RUNBOOK PLOT-01: processes are stacked in the paper's fixed order.
    RUNBOOK PLOT-03: the last bin absorbs the overflow, as the captions of
    Figures 4 and 5 state ("The last bin contains overflow events"). Without
    it our last bin falls off while the paper's rises, which is the single
    most visible shape difference in the comparison.
    """
    bins = np.linspace(x_range[0], x_range[1], n_bins + 1)
    centers = 0.5 * (bins[1:] + bins[:-1])
    hi_edge = bins[-1]

    def _hist(values, weights=None):
        v = np.asarray(values, dtype=float)
        if overflow:
            # Push everything above the last edge into the last bin. Clipping
            # just below the edge keeps np.histogram from dropping it.
            v = np.minimum(v, hi_edge - 1e-9)
        return np.histogram(v, bins=bins, weights=weights)[0]

    grouped_counts, grouped_info = {}, {}
    for name, df in dfs_dict.items():
        if name in DATA_SL:
            continue
        counts = _hist(df[var_name], weights=df["final_weight"])
        g, label, color = get_group_info(name)
        if g not in grouped_counts:
            grouped_counts[g] = counts
            grouped_info[g] = {"label": label, "color": color, "yield": counts.sum()}
        else:
            grouped_counts[g] += counts
            grouped_info[g]["yield"] += counts.sum()

    # PLOT-01: the paper's order, restricted to the groups actually present.
    order = [g for g in PAPER_STACK_ORDER if g in grouped_counts]
    order += [g for g in grouped_counts if g not in PAPER_STACK_ORDER]

    mc_counts = [grouped_counts[g] for g in order]
    mc_colors = [grouped_info[g]["color"] for g in order]
    mc_labels = [grouped_info[g]["label"] for g in order]
    total_mc = np.sum(mc_counts, axis=0) if mc_counts else np.zeros(n_bins)

    df_data = dfs_dict.get(channel_label)
    data_counts = (_hist(df_data[var_name]) if df_data is not None
                   else np.zeros(n_bins))

    fig, (ax, rax) = plt.subplots(
        2, 1, figsize=(9, 9), sharex=True,
        gridspec_kw={"height_ratios": [3, 1], "hspace": 0.05})

    if mc_counts:
        hep.histplot(mc_counts, bins=bins, stack=True, histtype="fill",
                     color=mc_colors, label=mc_labels,
                     edgecolor="black", linewidth=0.7, ax=ax)
    hep.histplot(data_counts, bins=bins, histtype="errorbar", color="black",
                 label="Data", yerr=True, marker="o", markersize=5, ax=ax)

    hep.cms.label("Preliminary", data=True, lumi=lumi_label, year=2016, ax=ax)
    ax.text(0.04, 0.90, region_name, transform=ax.transAxes,
            fontsize=17, fontweight="bold", va="top")

    # RUNBOOK PLOT-05: legend in the paper's reading order -- Data, then the
    # stack from the TOP down: Z(ll), Z(nunu), W(lnu), ttbar, t+X, VV/VH.
    # Relying on the order matplotlib hands back was giving the reverse
    # (bottom-up), so the legend disagreed with the stack it described. Built
    # explicitly from `order` instead, which cannot drift.
    handles, labels = ax.get_legend_handles_labels()
    by_label = OrderedDict(zip(labels, handles))
    wanted = ["Data"] + [grouped_info[g]["label"] for g in reversed(order)]
    leg_labels = [l for l in wanted if l in by_label]
    leg_labels += [l for l in by_label if l not in leg_labels]
    ax.legend([by_label[l] for l in leg_labels], leg_labels,
              fontsize=13, ncol=2, loc="upper right")
    ax.set_ylabel("Events", fontsize=20)
    ax.set_yscale("log")
    # RUNBOOK PLOT-04: the axis spans the paper's PROPORTIONS, not its absolute
    # range. Open Data gives us 2.4-4.9 fb^-1 against the paper's 35.9, so our
    # stacks sit roughly an order of magnitude lower; pinning the paper's
    # absolute limits would shove our histogram into the bottom of an empty
    # frame and make the comparison look worse than it is. Instead the limits
    # are fixed multiples of the stack peak, chosen to reproduce the decades
    # visible in Figures 4 and 5 (about 3.5 below the peak, 2 above it for the
    # legend). Every panel in a channel therefore has the same dynamic range.
    peak = max(data_counts.max(), total_mc.max(), 1.0)
    ax.set_ylim(peak / y_span[0], peak * y_span[1])

    with np.errstate(divide="ignore", invalid="ignore"):
        ratio = np.where(total_mc > 0, data_counts / total_mc, np.nan)
        rerr = np.where(total_mc > 0, np.sqrt(data_counts) / total_mc, np.nan)
    rax.errorbar(centers, ratio, yerr=rerr, fmt="o", color="black", markersize=5)
    rax.axhline(1.0, color="#888888", linestyle="--", linewidth=1)
    rax.set_ylim(0, 2.4)
    rax.set_ylabel("Data / Bkg", fontsize=14)
    rax.set_xlabel(x_label, fontsize=20)
    rax.set_xlim(x_range)
    rax.grid(axis="y", alpha=0.25)

    n_d, n_m = data_counts.sum(), total_mc.sum()
    if n_m:
        rax.text(0.98, 0.86, f"total Data/Bkg = {n_d / n_m:.2f}",
                 transform=rax.transAxes, ha="right", va="top", fontsize=11,
                 color="#444444")

    plt.tight_layout()
    plt.show()


In [ ]:
def run_analysis_for_channel(channel_mode):

    raw_dfs = load_data(channel_mode)
    if not raw_dfs:
        return

    regions_db = {"SR_1b_0f": {}, "SR_1b_1f": {}, "SR_2b": {}}
    for name, df in raw_dfs.items():
        for reg, df_reg in signal_regions(df).items():
            regions_db[reg][name] = df_reg

    d_label = "met" if channel_mode == "met" else "SingleMuon"

    # RUNBOOK PLOT-02: paper Figure 5 bins pTmiss from 250 to 550 GeV in
    # fifteen 20 GeV bins, the last one holding the overflow. We had 250-600
    # in ten 30 GeV bins, so no bin edge lined up with the paper's.
    kw = dict(n_bins=15, lumi_label=round(LUM / 1000.0, 1))

    plot_region_stack(regions_db["SR_2b"], "0l, 2 b tag, SR",
                      "met", r"$p_T^{miss}$ [GeV]", (250, 550), d_label, **kw)
    plot_region_stack(regions_db["SR_1b_1f"], "0l, SR, 1 FJ",
                      "met", r"$p_T^{miss}$ [GeV]", (250, 550), d_label, **kw)
    plot_region_stack(regions_db["SR_1b_0f"], "0l, SR, 0 FJ",
                      "met", r"$p_T^{miss}$ [GeV]", (250, 550), d_label, **kw)


In [ ]:

from collections import OrderedDict


if 'sum_weights_map' in globals():
    run_analysis_for_channel("met")
    #run_analysis_for_channel("electron")
else:
    print(" Error.")



> **Reference figure `Table_14.png` is not included in this repository.**
> It is a screenshot of a table from the paper and carries no physics; see `paper.pdf`.


In [ ]:

def filter_cr_wlnu(df):
    """
    Control Region: CR W(lν)
    Nombres oficiales Tabla 14:
        - SL1eWR  (electrón)
        - SL1mWR  (muón)
    """
    if df is None or len(df) == 0:
        return {}

    mask = (
        (df["met"] >= 250) &
        #(df["mT"] <= 160) &
        (df["nBTag"] == 0) &
        (df["nJet"] >= 3)
    )

    df_cr = df[mask]
    if len(df_cr) == 0:
        return {}

    return {"CR_Wlnu": df_cr}



Once `plot_region_stack` was defined, the same base structure was reused to create `plot_cr_stack`. The core stacked histogram logic stays the same, but additional lines were added or modified to specialize the plot for Control Regions.

The main changes are:

* A **new function name** was created: `plot_cr_stack`.
* A new input parameter was added to handle the **control region label** (for example: `region_label`).
* The plot now includes a **region tag drawn on the figure**, added through a line like:
  `ax.text(..., region_label, ...)`
* The function was adapted to work with **Control Region–filtered datasets**, instead of general or signal regions.
* The default axis ranges and binning were tuned for control-region kinematics.

All histogram stacking, coloring, grouping, and legend logic remains the same as in the previous functions.

In [ ]:

def plot_cr_stack(dfs_dict, region_label, var_name, x_label, x_range, data_label, n_bins=15):

    bins = np.linspace(x_range[0], x_range[1], n_bins + 1)
    grouped_counts = {}
    grouped_info = {}

    # A) MC
    for name, df in dfs_dict.items():
        if name in DATA_SL:
            continue

        counts, _ = np.histogram(df[var_name], bins=bins, weights=df["final_weight"])
        g_key, label, color = get_group_info(name)

        if g_key not in grouped_counts:
            grouped_counts[g_key] = counts
            grouped_info[g_key] = {"label": label, "color": color, "yield": np.sum(counts)}
        else:
            grouped_counts[g_key] += counts
            grouped_info[g_key]["yield"] += np.sum(counts)

    active_groups = sorted(grouped_info.keys(), key=lambda k: grouped_info[k]["yield"])

    mc_counts = []
    mc_colors = []
    mc_labels = []
    total_mc = np.zeros(n_bins)

    for g in active_groups:
        mc_counts.append(grouped_counts[g])
        mc_colors.append(grouped_info[g]["color"])
        mc_labels.append(grouped_info[g]["label"])
        total_mc += grouped_counts[g]

    # B) DATA
    df_data = dfs_dict.get(data_label)
    if df_data is not None:
        data_counts, _ = np.histogram(df_data[var_name], bins=bins)
    else:
        data_counts = np.zeros(n_bins)

    # === Plot ===
    fig, ax = plt.subplots(figsize=(10, 8))

    hep.histplot(mc_counts, bins=bins, stack=True, histtype="fill",
                 color=mc_colors, label=mc_labels, edgecolor="black", linewidth=1, ax=ax)

    hep.histplot(data_counts, bins=bins, histtype="errorbar", color="black",
                 label=f"{data_label} (Data)", yerr=True, marker='o', markersize=5, ax=ax)

    hep.cms.label("Preliminary", data=True, lumi=round(LUM / 1000.0, 1), year=2016, ax=ax)

    ax.text(0.05, 0.93, region_label, transform=ax.transAxes,
            fontsize=20, fontweight='bold', va='top', ha='left')

    handles, labels = ax.get_legend_handles_labels()
    by_label = OrderedDict(zip(labels[::-1], handles[::-1]))
    ax.legend(by_label.values(), by_label.keys(), fontsize=15, ncol=2, loc='upper right')

    ax.set_xlabel(x_label, fontsize=24)
    ax.set_ylabel("Events", fontsize=24)
    ax.set_xlim(x_range)
    ax.set_yscale("log")

    max_y = max(np.max(data_counts), np.max(total_mc))
    ax.set_ylim(0.1, max_y * 500)

    plt.tight_layout()
    plt.show()





In [ ]:


def run_SL(channel_mode):

    # 1. Load datasets
    raw_dfs = load_data(channel_mode)
    if not raw_dfs:
        return

    # 2. Filtrar región CR W(lν)
    cr_dict = {}
    for name, df in raw_dfs.items():
        reg = filter_cr_wlnu(df)
        if "CR_Wlnu" in reg:
            cr_dict[name] = reg["CR_Wlnu"]

    if len(cr_dict) == 0:
        print(" No events passed CR W(lν).")
        return

    # 3. Etiquetas correctas según Tabla 14
    if channel_mode == "muon":
        region_label = "SL1mWR"
        data_label = "SingleMuon"
    elif channel_mode == "electron":
        region_label = "AH1eWR"
        data_label = "SingleElectron"
    else:
        channel_mode == "met"
        region_label = "AH0lWR"
        data_label = "met"


    # plot MET
    plot_cr_stack(cr_dict, region_label, "met", r"$p_T^{miss}$ [GeV]", (250, 600), data_label, n_bins=15)

    # plot MT
    #plot_cr_stack(cr_dict, region_label, "mT_W", r"$m_T^W$ [GeV]", (160, 500), data_label, n_bins=15)


# --- RUN ---
#run_SL("muon")
#run_SL("electron")
run_SL("met")